# MLMarker tissue prediction — full pipeline

Predict tissue of origin for all runs in `pride_quant.parquet` using MLMarker (Random Forest, NSAF model).  
Only accept predictions with confidence >= 0.3 (99% accuracy on validation set).  
Output: `run_meta_mlmarker.tsv` with columns `pxd`, `run`, `tissue`, `confidence`.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from mlmarker import MLMarker
from mlmarker.utils import validate_sample

PROJECT_ROOT = Path(r"C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata")
CONFIDENCE_THRESHOLD = 0.3

# Load data
quant = pd.read_parquet(PROJECT_ROOT / "quant_data" / "pride_quant.parquet")
print(f"Loaded {len(quant)} samples from pride_quant.parquet")

# Initialize MLMarker
m = MLMarker()
protein_cols = [c for c in quant.columns if c not in ["pxd", "run", "source"]]
print(f"Model: {len(m.get_model_features())} features, {len(m.get_model_classes())} tissue classes")

Loaded 89626 samples from pride_quant.parquet
Model: 5979 features, 34 tissue classes


c:\Users\sander\miniconda3\envs\agentic-metadata\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\sander\miniconda3\envs\agentic-metadata\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
# Validate all samples at once (align columns to model features)
all_samples = quant[protein_cols]
validated = validate_sample(m.features, all_samples)
print(f"Validated matrix: {validated.shape}")

# Predict all at once
print("Running predict_proba...")
probabilities = m.model.predict_proba(validated)
classes = m.model.classes_
print(f"Done! Probabilities shape: {probabilities.shape}")

# Extract top-1 prediction and confidence per sample
top1_idx = probabilities.argmax(axis=1)
top1_tissue = classes[top1_idx]
top1_conf = probabilities[np.arange(len(probabilities)), top1_idx]

# Build results
results = pd.DataFrame({
    "pxd": quant["pxd"].values,
    "run": quant["run"].values,
    "tissue": top1_tissue,
    "confidence": np.round(top1_conf, 4),
})

print(f"\nAll predictions: {len(results)}")
print(f"Confidence stats: mean={results['confidence'].mean():.3f}, median={results['confidence'].median():.3f}")
print(f"Above threshold ({CONFIDENCE_THRESHOLD}): {(results['confidence'] >= CONFIDENCE_THRESHOLD).sum()} / {len(results)}")

Validated matrix: (89626, 5979)
Running predict_proba...
Done! Probabilities shape: (89626, 34)

All predictions: 89626
Confidence stats: mean=0.164, median=0.137
Above threshold (0.3): 5212 / 89626


In [6]:
# Filter to confident predictions and export
confident = results[results["confidence"] >= CONFIDENCE_THRESHOLD].copy()
confident = confident[["pxd", "run", "tissue", "confidence"]].sort_values(["pxd", "run"]).reset_index(drop=True)

print(f"Confident predictions (>= {CONFIDENCE_THRESHOLD}): {len(confident)} / {len(results)} ({len(confident)/len(results):.1%})")
print(f"\nTissue distribution:")
print(confident["tissue"].value_counts().to_string())

# Save
out_path = PROJECT_ROOT / "notebooks" / "run_meta_mlmarker.tsv"
confident.to_csv(out_path, sep="\t", index=False)
print(f"\nSaved to {out_path}")

Confident predictions (>= 0.3): 5212 / 89626 (5.8%)

Tissue distribution:
tissue
Brain              1998
Liver               781
Heart               694
Skeletal muscle     261
Prostate            249
Testis              217
Kidney              206
Monocytes           173
Ovary               138
Small intestine      83
Salivary gland       82
Placenta             68
Colon                65
Pituitary gland      51
Esophagus            45
Bone marrow          30
Tonsil               19
B-cells              15
Lung                 10
Stomach               7
Duodenum              6
Nasal Polyps          4
Adrenal gland         3
Thyroid               3
Urinary bladder       1
Endometrium           1
Oviduct               1
Adipose tissue        1

Saved to C:\Users\sander\OneDrive\Bureaublad\Projects\agentic-metadata\notebooks\run_meta_mlmarker.tsv
